# Managed vs External Tables — And Why Your File Got Deleted
**Day 4 | Run on `dev-cluster`**

---

## Why did the file get deleted from ADLS when you deleted it from the Volume UI?

The Volume is not a copy of your ADLS data — it is a **pointer** to it.

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
          maps directly to
abfss://bronze@evdatalakedev.dfs.core.windows.net/
```

Deleting a file through the Volume UI = deleting the real ADLS file. No recycle bin.

---

## Managed vs External — core difference

| | Managed | External |
|---|---|---|
| `LOCATION` in CREATE TABLE | No — UC picks the path | Yes — you specify your ADLS path |
| Who owns the files | Unity Catalog | You |
| `DROP TABLE` | metadata + **files deleted** | metadata deleted, **files safe** |
| `DROP SCHEMA CASCADE` | metadata + **files deleted** | metadata deleted, **files safe** |
| Delete file via Volume UI | **file deleted** | **file deleted** |
| `dbutils.fs.rm(path)` | **file deleted** | **file deleted** |
| Re-register after DROP | not possible | yes — same `LOCATION` |

> **Key rule:** Managed vs External only changes `DROP` behaviour. Deleting a file directly (UI or code) always touches real ADLS — for both types.

---

## Why `demo` container is used for the external table demo

`bronze`, `silver`, and `gold` containers all have Volumes registered on top of them.
Unity Catalog blocks `CREATE TABLE ... LOCATION` inside a Volume-governed path with error:
`Unsupported path operation PATH_CREATE_TABLE on volume`.

`demo` container has an External Location registered but **no Volume** on top — UC allows it as an external table `LOCATION`.

---

## Notebook structure

| Cell | What it does |
|---|---|
| 1 | Setup — set catalog and schema |
| 2 | Part A — create managed table, insert data |
| 3 | Part A — show where UC stored the files |
| 4 | Part A — DROP and prove files are gone |
| 5 | Part B — write Delta files to demo/ ADLS path |
| 6 | Part B — create external table with LOCATION |
| 7 | Part B — DROP and prove files survive |
| 8 | Part B — re-register from same files + cleanup |
| 9 | Part C — check bronze-volume type, explain UI delete |
| 10 | Part D — reference card |

In [ ]:
# Cell 1 — Setup
spark.sql("USE CATALOG dbw_ev_intelligence_dev")
spark.sql("USE SCHEMA default")
print("Catalog :", spark.sql("SELECT current_catalog()").collect()[0][0])
print("Schema  :", spark.sql("SELECT current_schema()").collect()[0][0])

In [ ]:
# Cell 2 — Part A: Create managed table and insert data
# No LOCATION = UC picks storage path. You do not control where files go.

spark.sql("""
    CREATE TABLE IF NOT EXISTS ev_stations_managed (
        station_id   STRING,
        station_name STRING,
        city         STRING,
        capacity_kw  INT
    ) USING DELTA
""")

spark.sql("""
    INSERT INTO ev_stations_managed VALUES
        ('ST001', 'MG Road Charger',    'Bengaluru', 150),
        ('ST002', 'Andheri Fast Charge','Mumbai',    200),
        ('ST003', 'Cyber Hub Station',  'Gurugram',  120)
""")

print("Managed table created and data inserted.")
spark.sql("SELECT * FROM ev_stations_managed").show()

In [ ]:
# Cell 3 — Part A: Find where UC stored the managed table files

detail = spark.sql("DESCRIBE DETAIL ev_stations_managed").collect()[0]
print("Table type :", detail['tableType'])   # MANAGED
print("Location   :", detail['location'])    # UC-managed path — NOT your abfss://bronze@...
print("Format     :", detail['format'])
print()
print("UC owns these files. DROP TABLE will permanently delete them from ADLS.")

In [ ]:
# Cell 4 — Part A: DROP managed table and prove files are deleted

location_before = spark.sql("DESCRIBE DETAIL ev_stations_managed").collect()[0]['location']
print("Files were at:", location_before)

spark.sql("DROP TABLE ev_stations_managed")
print("Table dropped.")
print()

try:
    dbutils.fs.ls(location_before)
    print("Files still exist — unexpected.")
except Exception:
    print("FILES GONE — confirmed deleted by DROP.")
    print("LESSON: DROP TABLE on MANAGED = metadata gone + ADLS files deleted. No recovery.")

In [ ]:
# Cell 5 — Part B: Write Delta files to demo/ ADLS path
# demo/ has no Volume on top — allowed as external table LOCATION.
# bronze/, silver/, gold/ all have Volumes registered so UC blocks CREATE TABLE there.

from pyspark.sql import Row

EXTERNAL_PATH = "abfss://demo@evdatalakedev.dfs.core.windows.net/ev_stations_external/"

rows = [
    Row(station_id="ST004", station_name="Koramangala Hub",    city="Bengaluru", capacity_kw=180),
    Row(station_id="ST005", station_name="Bandra Supercharge", city="Mumbai",    capacity_kw=250),
    Row(station_id="ST006", station_name="DLF Cyber City",     city="Gurugram",  capacity_kw=100),
]
spark.createDataFrame(rows).write.format("delta").mode("overwrite").save(EXTERNAL_PATH)

print(f"Data written to: {EXTERNAL_PATH}")
files = dbutils.fs.ls(EXTERNAL_PATH)
print(f"{len(files)} item(s) at path:")
for f in files:
    print(f"  {f.path}")

In [ ]:
# Cell 6 — Part B: Create external table pointing to that path
# LOCATION keyword = what makes a table EXTERNAL.

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS ev_stations_external
    USING DELTA
    LOCATION '{EXTERNAL_PATH}'
""")

detail = spark.sql("DESCRIBE DETAIL ev_stations_external").collect()[0]
print("Table type :", detail['tableType'])   # EXTERNAL
print("Location   :", detail['location'])    # your abfss://silver@evdatalakedev...
print()
print("UC stores only metadata. Files stay on YOUR ADLS path.")
spark.sql("SELECT * FROM ev_stations_external").show()

In [ ]:
# Cell 7 — Part B: DROP external table and prove files survive

print(f"Files BEFORE drop: {len(dbutils.fs.ls(EXTERNAL_PATH))} item(s)")

spark.sql("DROP TABLE ev_stations_external")
print("Table dropped from Unity Catalog.")
print()

try:
    spark.sql("SELECT * FROM ev_stations_external").show()
except Exception:
    print("Query failed (expected) — table no longer in UC.")

print(f"Files AFTER drop: {len(dbutils.fs.ls(EXTERNAL_PATH))} item(s) — FILES PRESERVED on ADLS.")
print()
print("LESSON: DROP TABLE on EXTERNAL = metadata gone, files safe.")
print("        Re-register anytime with the same LOCATION — no data loss.")

In [ ]:
# Cell 8 — Part B: Re-register from same files, then cleanup

spark.sql(f"""
    CREATE TABLE ev_stations_external
    USING DELTA
    LOCATION '{EXTERNAL_PATH}'
""")
print("Table re-registered — data back instantly, no reload needed.")
spark.sql("SELECT * FROM ev_stations_external").show()

spark.sql("DROP TABLE IF EXISTS ev_stations_external")
dbutils.fs.rm(EXTERNAL_PATH, recurse=True)
print("Cleanup done.")

In [ ]:
# Cell 9 — Part C: Check bronze-volume type and explain the UI delete

try:
    for row in spark.sql("DESCRIBE VOLUME dbw_ev_intelligence_dev.default.`bronze-volume`").collect():
        print(f"  {row[0]:20s} : {row[1]}")
except Exception as e:
    print(f"Could not describe volume: {e}")

print()
print("Volume UI = file browser for your real ADLS path.")
print("Delete via UI  →  real ADLS file deleted immediately. No recycle bin.")
print("Applies to both Managed and External Volumes.")
print()
print("The ONLY action that does NOT touch ADLS files:")
print("  DROP TABLE / DROP VOLUME on an EXTERNAL object.")

In [ ]:
# Cell 10 — Part D: Reference card

print(f"  {'Action':<38} {'Managed':<20} {'External'}")
print("  " + "-" * 70)
rows = [
    ("DROP TABLE",                   "meta + FILES deleted", "meta deleted, FILES SAFE"),
    ("DROP SCHEMA CASCADE",          "meta + FILES deleted", "meta deleted, FILES SAFE"),
    ("DROP VOLUME",                  "meta + FILES deleted", "meta deleted, FILES SAFE"),
    ("TRUNCATE TABLE",               "all rows deleted",     "all rows deleted"),
    ("DELETE FROM table WHERE ...",  "rows deleted",         "rows deleted"),
    ("Delete file via Volume UI",    "file deleted",         "file deleted"),
    ("dbutils.fs.rm(path)",          "file deleted",         "file deleted"),
    ("Re-register after DROP TABLE", "not possible",         "yes — same LOCATION"),
]
for action, managed, external in rows:
    print(f"  {action:<38} {managed:<20} {external}")
print()
print("  MANAGED  → temp tables, Databricks-only data")
print("  EXTERNAL → Bronze/Silver/Gold tables (ADF + Synapse + Power BI share same ADLS files)")